### Tutorial Pandas y SQLite

##### Traemos las librerías necesarias

In [ ]:
from sqlalchemy import create_engine
import pandas as pd

##### La base de datos y el motor (engine)

In [ ]:
# Nombre de la base de datos
db_file = 'tutorial.db'

# Ruta de dónde quieres guardar la base de datos
db_path = './data/'

# Motor con el que SQLAlchemy va a trabajar. En este tutorial usaremos SQLite.
engine = create_engine(f'sqlite:///{db_path+db_file}')

engine

!Ya tienes el engine listo! El engine es la manera en cómo te vas a comunicar con la base de datos, para leer o escribir en ella.

¿Te has fijado que todavía no existe 'tutorial.db'? Vamos a ello

In [ ]:
# Con esta linea te conectas al engine, esto hace que, si no existe ya la base de datos 
# (el archivo tutorial.db), se cree automáticamente.
engine.connect()

In [ ]:
# Con esta linea te desconectas del engine, es muy importante que en cada bloque "with" que
# uses para conectarte al engine termines con esta línea. 
# Si te quedas conectado a la db puedes generar problemas de bloqueos, agotamiento de conexción y fugas de memoria.
engine.dispose()

En la práctica, no creamos la .db vacía primero para luego usar un bloque "with". Simplemente
generamos la .db al mismo tiempo que se usa "with" por primera vez... no perdemos el tiempo.

##### DataFrame básico

Para este tutorial ocupamos crear un "DataFrame" en pandas. Crearemos uno básico.

In [ ]:
data = {
    'numbers':[1,2,3,4,5,6,7,8,9,10],
    'letter':['a','b','c','d','e','f','g','h','i','j']
    }

df = pd.DataFrame(data)
df

##### Escribiendo en la .db "El bloque With"

Guardar un DataFrame en la .db

In [ ]:
# Siempre usamos el bloque "with", ya sea para uno o varias instrucciones.
# Recuerda pasar index=False para que no se genere una columna extra para los índices que usa
# pandas en sus dataframes.

with engine.connect() as conn, conn.begin():  
    df.to_sql(name = 'mi_primer_tabla', index= False, con = conn, if_exists = 'replace')
    # df.to_sql(name = 'otro_df', index= False, con = conn)
    # Otra instrucción (leer o escribir, da igual)

engine.dispose()

En el explorador de VsCode, ve al directorio donde se guardó "tutorial.db" y da click derecho en el archivo, despues da click en la opción "OpenDatabase". En la parte inferior se va a abrir un apartado "SQLite Explorer" donde podras visualizar tus tablas (recuerda antes descargar la extensión de SQLite para VsCode, si no esto no va a funcionar).

##### Leyendo en la .db "El bloque With"

Con la base de datos creada en este tutorial ("tutorial.db") y la tabla que acabas de ingresar ("mi_primer_tabla") vamos a hacer lecturas y las guardaremos en otros dataframes

Crear un DataFrame desde una tabla en una base de datos SQL

In [ ]:
# Esta función sirve para traerte toda la tabla

with engine.connect() as conn, conn.begin():  
    df_tabla = pd.read_sql_table('mi_primer_tabla', conn)
    # Otra instrucción (leer o escribir, da igual)

engine.dispose()

df_tabla

Crear un DataFrame desde un query en una base de datos SQL

In [ ]:
# Esta función sirve para traerte un query

query = """ SELECT * FROM mi_primer_tabla WHERE numbers <= 5 """

with engine.connect() as conn, conn.begin():
    df_query = pd.read_sql(query, conn)
    # Otra instrucción (leer o escribir, da igual)

engine.dispose()

df_query